# V2 Benchmark出题：候选设计与原图检索

最终知识 → 一次候选设计 →（仅编辑：搜原图、核验、定稿）→ 校验与审核 → 作答检索 → 导出。

**输入不再包含手写plan、intent、选材ID或预先匹配的目标图。**全局seed、预算、题型、模型及声明的原图检索源属于运行配置。任务由知识、条件与视觉推理支撑；简单直接的知识应用也可成立，明确执行要求只作辅助，以新题实际检查信号、题面和原判据的对应，不要求training-free提升。未发布知识不会进入模型；配图、编辑原图、监督目标分别管理。已有知识本身仍有质量限制，因此新产物称开发候选。

从下面第一个代码cell开始；默认只读实际保存结果。所有展示使用Markdown、表格和原生图片，不使用HTML。逐步输入／输出与prompt见 [benchmark/stepbystep.ipynb](stepbystep.ipynb)。



**当前按指导信号与训练候选思路迭代。** 新题用于检查材料、任务、公开展示条件与原题判据的对应；失败可用于研究训练，但不等于正确监督已备齐。作答检索与评测共享公开题面检索实现。旧案例仍按冻结版本解释；新版执行使用新的NEW_RUN。

当前主要prompt：[候选设计](prompts/design_candidates.md)、[原图审核（本地与外搜共用）](prompts/select_edit_source.md)、[编辑定稿](prompts/construct.md)、[任务审核](prompts/review_task.md)。逐步版每个模型算子前展示当前prompt全文；旧案例的实际输入以其冻结请求为准。

In [ ]:
from curation.preparation.records import rows
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")]
PROJECT = next((p for p in _candidates if (p / "curation/benchmark/authoring.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("找不到包含curation的项目根目录")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
try:
    from curation.preparation.inspection import show_records, show_summary, show_cases, snapshot_ref
    from curation.benchmark.runtime import config as build_config
    from curation.preparation.records import run_records, run_state, rows
except ModuleNotFoundError as error:
    raise RuntimeError("请选择demiwtg内核：/yzp/zhaozy/yangzepeng/0905/env/bin/python") from error

MODE = "view_saved"  # execute运行；view_saved只读已有checkpoint。
BASE = PROJECT / "curation/benchmark/runs"
SAVED_RUN = BASE / "pipeline_v2_benchmark_review"
NEW_RUN = PROJECT / "curation/benchmark/runs/pipeline_v2_benchmark_review"  # 输入／代码／prompt变化须新run。
knowledge_runs = []  # 填入知识发布 ID 或固定 DatasetRef；视觉发布可另传 visual_runs。
# 知识输入为最终交付；编辑原图检索的全库清单及外部源在下面config中公开配置。
# 发现和选题只读最终知识；原始场景／目标图片仅在后续显式素材步骤按需读取。
MODEL_BACKEND = "offline"  # local用本地Qwen；offline为相同prompt／context生成绑定请求。
# concepts=None使用全部发布概念；可传名称列表做可复现小批。
config = build_config(MODEL_BACKEND, concepts=None, seed=0, max_units=2, tasks_per_unit=1, max_context_chars=100000,
    scene_search={"image_ref": None,
                  "external_providers": ["commons"]},
    author_model="gpt-6-astra" if MODEL_BACKEND == "offline" else None,
    author_effort="high" if MODEL_BACKEND == "offline" else None)
CASE_ID = None  # 默认展示本阶段第一条实际参与处理的记录；也可填unit_id或task_id。
SHOW_AUDIT = False  # 完整来源、字段、实际模型请求；图片按角色原样展示。
THROUGH = "export"
if MODE not in {"view_saved", "execute"}:
    raise ValueError("MODE必须是view_saved或execute")
run = SAVED_RUN if MODE == "view_saved" else NEW_RUN
if MODE == "view_saved":
    saved = run_records(run).get("manifest")
    if saved is not None:
        config = saved["config"]
    else:
        print("请指定已有 V2 Lance run；后续查看单元需要已完成的阶段。")
print("项目：", PROJECT, "\n内核：", sys.executable, "\n模式：", MODE, "\n运行：", run)


print("本次运行配置：", config)


## 完整算子链

| 顺序 | 算子 | 输入 → 输出 |
|---|---|---|
| 1. 读取最终知识 | `map(ReadKnowledge)` | 已发布的知识发布 DatasetRef → 每概念一份完整知识包 |
| 2. 一次设计候选任务 | `filter（公开seed预算）→ map_prompt_async("design_candidates")` | 完整知识包＋全局预算 → 候选考点与任务设计 |
| 3. 展开候选与绑定证据 | `flat_map(ExpandCandidates)` | 模型候选列表 → 每题一行 |
| 4. 编辑：按初始场景搜本地图 | `map_async(SearchLocalScenes)` | 编辑原图需求及查询词＋全库图片清单 → 候选原图 |
| 5. 仅编辑：核验原图 | `map_prompt_async("select_edit_source")` | 知识考点、最终知识、候选原图像素与来源 → 合格原图或needs_edit_source |
| 6. 编辑：必要时外部补搜 | `map_async(SearchExternalScenes)` | 本地无候选或像素审核拒绝 → 外部候选或可追溯缺口 |
| 7. 编辑：核验外部原图 | `map_prompt_async("select_edit_source_external")` | 外部候选像素及来源 → 可用原图或缺口 |
| 8. 编辑：据真实原图定稿 | `map_prompt_async("construct")` | 候选设计＋选定原图像素 → 具体题面和判据 |
| 9. 校验任务契约 | `map(ValidateTask)` | 草稿和材料编号 → 绑定稳定知识ID的判据及冻结任务哈希 |
| 10. 审核任务质量 | `map_prompt_async("review_task")` | 草稿、全部构题证据、编辑原图 → 五项语义审核 |
| 11. 按题面检索作答参考 | `map(RetrieveForAnswer)` | 公开instruction与共享知识表 → 实际作答材料及检索差距 |
| 12. 导出并保留缺口 | `map(export_record) → filter` | 任务及各阶段审核 → ready／incomplete |

下面就是CLI实际执行的函数；map负责业务，map_prompt_async负责模型调用，checkpoint保存各边界。offline按同一请求生成待响应状态；不暗中继承本次聊天。

In [ ]:
from functools import partial
from demiflow.standalone import local_data
from curation.preparation.records import run_lock
from curation.benchmark.runtime import graph_version
from curation.benchmark.authoring import AuthoringRunFiles, input_records, split_guard, knowledge_items
from curation.benchmark.candidates import ReadKnowledge, ExpandCandidates
from curation.benchmark.scene_search import SceneSearch, SearchLocalScenes, SearchExternalScenes
from curation.benchmark.operators import ValidateTask, RetrieveForAnswer, export_record
from curation.benchmark.prompting import prompt_config, prompt_responses, prepare_design, apply_design, prepare_external_edit_source, apply_external_edit_source, prepare_edit_source, apply_edit_source, prepare_construct, apply_construct, prepare_task_review, apply_task_review



In [ ]:
def run_pipeline(run, knowledge_runs, config, through="export", visual_runs=None):
    """Knowledge-first authoring; the notebook is the CLI graph source."""
    with run_lock(run):
        # 文章库与独立视觉库按概念组合（共享 publication_sources catalog）。
        files = AuthoringRunFiles(run, knowledge_runs, "benchmark", config, graph_version("benchmark"), visual_runs=visual_runs)
        guard = split_guard(files)
        pack, options = prompt_config(run, config)
        data = local_data(prompt_packs={"tasks.yaml": pack}, prompt_options=options,
                          max_prompt_requests=config["model"]["max_calls"])
        knowledge = (data.from_iter(lambda: input_records(files))
            .map(ReadKnowledge(files.knowledge_version, "benchmark", config, guard))
)
        knowledge = files.lance_checkpoint(knowledge, "knowledge")
        if through == "knowledge":
            return files.finish()
        scope = knowledge.filter(lambda r: r["status"] == "knowledge_available")
        # Author scope must not shrink the shared answer-retrieval catalog.
        if config.get("concepts"):
            scope = scope.filter(lambda r: r["concept"] in config["concepts"])
        if config["max_units"] is not None:
            ranks = scope.map(lambda r: (r["sampling_key"], r["unit_id"])).take_all()
            selected_ids = {unit for _, unit in sorted(ranks)[:config["max_units"]]}
            scope = scope.filter(lambda r: r["unit_id"] in selected_ids)
        designed = (scope
            .map(partial(prepare_design, run=run, pack=pack))
            .map_prompt_async("design_candidates", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="design_candidates_result", call_output="design_candidates_call", error_output="design_candidates_error",
                when=lambda r: r["status"] == "knowledge_available", concurrency=1, queue_depth=1)
            .map(partial(apply_design, run=run))
)
        designed = files.lance_checkpoint(designed, "design", extra=prompt_responses(run, "design_candidates"))
        if through == "design":
            return files.finish()
        candidates = (designed.flat_map(ExpandCandidates(guard, config))
)
        candidates = files.lance_checkpoint(candidates, "candidates")
        if through == "candidates":
            return files.finish()
        scene_search = SceneSearch(run, knowledge.flat_map(knowledge_items).take_all(), guard, config)
        edit_candidates = (candidates.map_async(SearchLocalScenes(scene_search), concurrency=1)
)
        edit_candidates = files.lance_checkpoint(edit_candidates, "edit_search")
        if through == "edit_search":
            return files.finish()
        source_ready = (edit_candidates
            .map(partial(prepare_edit_source, run=run, pack=pack))
            .map_prompt_async("select_edit_source", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="select_edit_source_result", call_output="select_edit_source_call", error_output="select_edit_source_error",
                when=lambda r: r["status"] == "edit_candidates_ready", concurrency=1, queue_depth=1)
            .map(partial(apply_edit_source, run=run))
)
        source_ready = files.lance_checkpoint(source_ready, "edit_source", extra=prompt_responses(run, "select_edit_source"))
        if through == "edit_source":
            return files.finish()
        external_candidates = (source_ready.map_async(SearchExternalScenes(scene_search), concurrency=1)
)
        external_candidates = files.lance_checkpoint(external_candidates, "edit_external_search")
        if through == "edit_external_search":
            return files.finish()
        all_sources = (external_candidates
            .map(partial(prepare_external_edit_source, run=run, pack=pack))
            .map_prompt_async("select_edit_source_external", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="select_edit_source_external_result", call_output="select_edit_source_external_call", error_output="select_edit_source_external_error",
                when=lambda r: r["status"] == "external_edit_candidates_ready", concurrency=1, queue_depth=1)
            .map(partial(apply_external_edit_source, run=run))
)
        all_sources = files.lance_checkpoint(all_sources, "edit_external_source", extra=prompt_responses(run, "select_edit_source_external"))
        if through == "edit_external_source":
            return files.finish()
        constructed = (all_sources
            .map(partial(prepare_construct, run=run, pack=pack))
            .map_prompt_async("construct", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="construct_result", call_output="construct_call", error_output="construct_error",
                when=lambda r: r["status"] == "selected", concurrency=1, queue_depth=1)
            .map(partial(apply_construct, run=run))
)
        constructed = files.lance_checkpoint(constructed, "construct", extra=prompt_responses(run, "construct"))
        if through == "construct":
            return files.finish()
        validated = (constructed.map(ValidateTask(guard))
)
        validated = files.lance_checkpoint(validated, "validate")
        if through == "validate":
            return files.finish()
        reviewed = (validated
            .map(partial(prepare_task_review, run=run, pack=pack))
            .map_prompt_async("review_task", config="tasks.yaml",
                inputs={"instructions": "prompt_instructions", "payload": "prompt_payload", "images": "prompt_images"},
                output="review_task_result", call_output="review_task_call", error_output="review_task_error",
                when=lambda r: r["status"] == "valid_task", concurrency=1, queue_depth=1)
            .map(partial(apply_task_review, run=run))
)
        reviewed = files.lance_checkpoint(reviewed, "review", extra=prompt_responses(run, "review_task"))
        if through == "review":
            return files.finish()
        retrieved = (reviewed.map(RetrieveForAnswer(knowledge.flat_map(knowledge_items).take_all(), guard))
)
        retrieved = files.lance_checkpoint(retrieved, "retrieve")
        if through == "retrieve":
            return files.finish()
        exported = (retrieved.map(export_record)
)
        exported = files.lance_checkpoint(exported, "export")
        ready = (exported.filter(lambda r: r["export_ready"])
)
        ready = files.lance_checkpoint(ready, "ready")
        incomplete = (exported.filter(lambda r: not r["export_ready"])
)
        incomplete = files.lance_checkpoint(incomplete, "incomplete")
        return files.finish()


## 执行与运行概览

In [ ]:
if MODE == "execute":
    import asyncio
    state = await asyncio.to_thread(run_pipeline, run, knowledge_runs, config, through=THROUGH)
show_summary(run)


## 实际候选与保留的失败

候选设计、题目、搜索记录、图片角色和审核逐条呈现；未入选概念可回knowledge查看，不混成坏题。

In [ ]:
show_cases(run)


## 完整请求与原始记录

In [ ]:
if SHOW_AUDIT:
    state = run_state(run)
    last = "export" if "export" in state["stages"] else next(reversed(state["stages"]))
    show_records(local_data().from_iter(lambda: rows(snapshot_ref(run, last))), last, CASE_ID, audit=True)
